# BGE-M3 hybrid + BGE-reranker: H100 reproducibility run

**Single experiment**, end-to-end, on a rented H100 / H200 / A100 80 GB instance.

## What this notebook does

Runs **one** experiment that the local Apple Silicon machine cannot finish in a reasonable wall-clock:

```
python3 scripts/run_experiment.py --method hybrid --embedding bge_m3 --reranker bge --top-k 20
```

It produces `data/results/hybrid+bge_bge_m3_whole_doc.json` (23,088 per-query records).

## What this notebook does NOT do

* Run any other experiment. The other two BGE-M3 configs (dense, hybrid no-rerank) already exist in `data/results/` from a prior local run; they are valid and are reused as-is.
* Generate figures or aggregate Markdown. Those run **locally on the laptop**, not on the rented GPU. See `scripts/aggregate_bge_m3.py` and `scripts/bge_m3_stats.py`.
* Edit the paper. That's a manual pass against `paper_neurips/main.tex` after results land.

## Expected wall-clock on H100 80 GB

* Dataset download: 2–5 min (one-time)
* BGE-M3 corpus embedding: 2–4 min (skipped on re-run if cache exists)
* Sanity run (100 queries): 1–2 min
* Full run (23,088 queries × ~20 candidates × cross-encoder): **30–60 min**
* **Total: ~60–90 min including HF cold cache.**

## Resume policy

Cells are idempotent. If a run dies mid-way, re-execute from the failing cell — the FAISS index is cached so re-embedding is skipped.


## 1. Environment check

Hard-fail if there's no CUDA. The point of this notebook is the H100; running it on CPU would take days.

In [ ]:
import sys, platform
import torch

assert torch.cuda.is_available(), 'CUDA is not available. This notebook requires a GPU instance.'
gpu = torch.cuda.get_device_name(0)
print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('Torch       :', torch.__version__)
print('CUDA        :', torch.version.cuda)
print('GPU         :', gpu)
print('GPU memory  : {:.1f} GB'.format(torch.cuda.get_device_properties(0).total_memory / 1e9))

if not any(x in gpu for x in ('H100', 'H200', 'A100')):
    print('\nWARNING: GPU is not H100/H200/A100. Batch sizes in configs/default.yaml are tuned for these; smaller cards may OOM.')

## 2. Repo setup

If you're running this on a fresh instance, the repo needs to be cloned and the dependencies installed.
If you've already done that (e.g. via `git clone` + `pip install -r requirements-h100.txt && pip install -e .` from a shell), skip this cell.

Adjust `REPO_URL` if your fork lives elsewhere.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/kaanrkaraman/rag-research-paper.git'
BRANCH   = 'fix/camera-ready-figures-and-refs'  # adjust to the branch you want to run
WORKDIR  = Path.home() / 'RAGPaper'

if not (WORKDIR / 'pyproject.toml').exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}

%cd {WORKDIR}
!pip install -q -r requirements-h100.txt
!pip install -q -e .
print('OK: repo ready at', WORKDIR)

## 3. Dataset

Download T²-RAGBench from HuggingFace and verify counts. The first call caches under `data/raw/`; re-runs hit the cache.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))

from src.data_loader import load_t2ragbench

data = load_t2ragbench()
assert data.num_queries == 23088, f'Expected 23088 queries, got {data.num_queries}'
assert data.num_documents == 7318, f'Expected 7318 documents, got {data.num_documents}'
print(data.summary())

## 4. Sanity run (100 queries)

Builds the FAISS index from scratch (or loads from cache) and runs the full pipeline on 100 queries. Should take 1–2 min on H100. Use the printed metrics to sanity-check that the reranker is doing its job before kicking off the full run.

In [ ]:
!python3 scripts/run_experiment.py \
    --method hybrid --embedding bge_m3 --reranker bge \
    --top-k 20 --max-queries 100 \
    --output-name sanity_hybrid+bge_bge_m3

In [ ]:
import json
sanity = json.load(open('data/results/sanity_hybrid+bge_bge_m3.json'))
print('config:', sanity['config'])
print('metrics (n=100):')
for k in ['recall@1', 'recall@5', 'recall@10', 'mrr@3', 'ndcg@10']:
    if k in sanity['retrieval_metrics']:
        print(f'  {k}: {sanity["retrieval_metrics"][k]:.4f}')
assert len(sanity['per_query_results']) == 100, 'sanity run did not produce 100 per-query records'
print('\nOK: pipeline runs end-to-end. Proceed to the full run.')

## 5. Full run (23,088 queries) — the actual experiment

**Wall-clock: 30–60 min on H100.** Output: `data/results/hybrid+bge_bge_m3_whole_doc.json`.

**Important:** `--top-k 20` is required. The CLI default of 5 silently caps every @K metric for K>5 (this bug cost a prior run). All other defaults come from `configs/default.yaml`.

In [ ]:
import time
t0 = time.time()
!python3 scripts/run_experiment.py \
    --method hybrid --embedding bge_m3 --reranker bge \
    --top-k 20
print(f'\nFull run elapsed: {(time.time()-t0)/60:.1f} min')

## 6. Provenance + result hashes

The result JSON's `config.provenance` block already contains git SHA, library versions, GPU name, HF model revision, and the FAISS index SHA-256 (set by `collect_provenance` in `src/utils/common.py`).

Below: dump pip freeze + a quick summary, save to `provenance.log` for archiving.

In [ ]:
result_file = 'data/results/hybrid+bge_bge_m3_whole_doc.json'
result = json.load(open(result_file))

print('=== Provenance ===')
for k, v in result['config']['provenance'].items():
    print(f'  {k}: {v}')

print('\n=== Headline metrics ===')
for k in ['recall@1', 'recall@5', 'recall@10', 'recall@20', 'mrr@3', 'ndcg@10', 'map']:
    if k in result['retrieval_metrics']:
        print(f'  {k}: {result["retrieval_metrics"][k]:.4f}')

print(f'\nper-query records: {len(result["per_query_results"])}')
print(f'avg latency:       {result["avg_latency_ms"]:.1f} ms/query')

# Save full pip freeze for archival reproducibility.
!pip freeze > provenance.log
print('\npip freeze saved to provenance.log')

## 7. Sync results back

Two options to get the result JSON off the rented instance:

**Option A — git push** (recommended; preserves history)

```bash
git checkout -b bge-m3-h100-runs
git add data/results/hybrid+bge_bge_m3_whole_doc.json provenance.log
git commit -m "Add BGE-M3 hybrid+BGE rerank results from H100"
git push origin bge-m3-h100-runs
```

**Option B — rsync to local laptop** (faster if you have the host)

```bash
# from your laptop, NOT this notebook
rsync -av <h100-host>:~/RAGPaper/data/results/hybrid+bge_bge_m3_whole_doc.json ./data/results/
rsync -av <h100-host>:~/RAGPaper/provenance.log ./
```

Once the JSON is on your laptop, follow `notebooks/AFTER_H100.md` (or the post-H100 paper-edit guide) to:
1. Run `python3 scripts/aggregate_bge_m3.py --out paper_neurips/bge_m3_results.md`
2. Run `python3 scripts/bge_m3_stats.py --out paper_neurips/bge_m3_results.md`
3. Run `python3 scripts/generate_figures.py` to update `paper_neurips/figures/`
4. Apply paper edits (see the post-H100 guide for the full list)
5. Recompile `paper_neurips/main.tex`.
